# Data understanding: what is in Teradata, and what the UI calls

**Purpose.** Everything in this project lives in Teradata: the transactions, the fraud model, the scores the dashboard
shows, and the policy text the copilot searches. This notebook walks through it step by step, and **every number below is
queried live from Teradata** when you run it, so you can check it yourself.

| Step | Question it answers |
|---|---|
| 1 | Am I connected to the right database? |
| 2 | Which tables and views exist, how big are they, and what are they for? |
| 3 | What does the transaction data look like? |
| 4 | How do raw transactions become model features? |
| 5 | Where is the model, and where are its scores? |
| 6 | What does each part of the UI call in Teradata? (the real SQL, captured while the app code runs) |
| 7 | Consistency checks across all of the above |

Read it top to bottom. Companion notebooks: `Model_Understanding.ipynb` (how the model was trained and whether it is
over/under-fitting) and `Data_Embedding.ipynb` (how the PDF was cleaned, embedded and searched).

## Step 1. Connect to Teradata
The app keeps **one** connection (see `service/db.py`). This notebook opens the same one with the credentials in `backend/.env`.

In [1]:
import os, sys, json, re, time, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# Run from backend/notebook (the folder of this notebook) or from backend/
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebook" else Path.cwd().resolve()
sys.path[:0] = [str(ROOT), str(ROOT / "scripts")]

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")            # TD_HOST / TD_USER / TD_PASSWORD / OPENAI_API_KEY (never written in the notebook)

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 110)
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False})

from common import connect             # opens the one Teradata connection used by the app (scripts/common.py)
connect()
from service import db                 # the app's own data layer: db.query_df(sql) -> pandas DataFrame

def q(sql):
    """Run a SELECT inside Teradata and get the result as a pandas DataFrame (same call the API uses)."""
    return db.query_df(sql)

print("connected as:", q("SELECT USER AS usr, DATABASE AS db").iloc[0].to_dict())
print("Teradata version:", q("SELECT InfoData FROM DBC.DBCInfoV WHERE InfoKey = 'VERSION'").iloc[0, 0])

connected as: {'usr': 'DEMO_USER', 'db': 'DEMO_USER'}
Teradata version: 20.00.24.60


## Step 2. Inventory: every table and view the app uses

All of these sit in the same Teradata database (the schema of the user shown above). The *purpose* column says who writes
each one and who reads it.

In [2]:
PURPOSE = {
    "txn":             "200,000 synthetic bank transactions + the fraud label. Written by scripts/load_data.py; read by every dashboard query.",
    "txn_features":    "VIEW: the numeric model inputs derived from txn (one-hot channel, high-risk merchant flag, ...). Read by XGBoost / XGBoostPredict.",
    "fraud_xgb_model": "THE MODEL: the trained in-database XGBoost, one row per decision tree (the whole tree is stored as JSON). Written by the training notebook/script.",
    "model_metrics":   "One row with how the model was trained (AUC, row counts, seconds). Read by GET /api/model (the dashboard model card).",
    "txn_scores":      "THE SCORES: fraud probability (Prob_1) for every transaction. Written by XGBoostPredict; read by alerts, KPIs and chat.",
    "policy_docs":     "One row per uploaded document (the PDF). Read/written by the Documents page.",
    "policy_chunks":   "The PDF split into searchable passages (text + page label). Read by the copilot.",
    "policy_emb":      "One 384-number embedding per passage. Searched IN Teradata with TD_VectorDistance.",
    "q_emb":           "Scratch table holding the embedding of the question being asked right now (emptied after each search).",
}
objects = q("SELECT TableName, TableKind FROM DBC.TablesV WHERE DatabaseName = DATABASE ORDER BY TableName")
objects = objects[objects.TableName.isin(PURPOSE)]
rows = []
for name, kind in zip(objects.TableName, objects.TableKind):
    n = int(q(f"SELECT COUNT(*) AS n FROM {name}")["n"][0])
    rows.append({"object": name, "kind": {"T": "table", "V": "VIEW"}.get(kind, kind), "rows": f"{n:,}", "purpose": PURPOSE[name]})
inventory = pd.DataFrame(rows)
print(inventory.to_string(index=False, max_colwidth=100))
assert set(PURPOSE) - {"q_emb"} <= set(inventory.object), "an expected table is missing"

         object kind    rows                                                                                              purpose
fraud_xgb_model   O      401 THE MODEL: the trained in-database XGBoost, one row per decision tree (the whole tree is stored a...
  model_metrics   O        1 One row with how the model was trained (AUC, row counts, seconds). Read by GET /api/model (the da...
  policy_chunks   T      287                     The PDF split into searchable passages (text + page label). Read by the copilot.
    policy_docs   T        2                         One row per uploaded document (the PDF). Read/written by the Documents page.
     policy_emb   T      287                   One 384-number embedding per passage. Searched IN Teradata with TD_VectorDistance.
          q_emb   T        1 Scratch table holding the embedding of the question being asked right now (emptied after each sea...
            txn   T  200,000 200,000 synthetic bank transactions + the fraud label. Writte

## Step 3. The transaction data (`txn`)
The data is **synthetic** (generated by `scripts/generate_data.py` with a fixed random seed, then loaded with `fastload`),
so every feature tells a business story (new device, foreign card, crypto merchant, 3am...). The fraud label is generated
from those signals plus noise, which is why a model can learn it but never perfectly.

In [3]:
cols = q("""SELECT ColumnName AS col, ColumnType AS type_code, ColumnLength AS len
            FROM DBC.ColumnsV WHERE DatabaseName = DATABASE AND TableName = 'txn' ORDER BY ColumnId""")
TYPES = {"I": "INTEGER", "I8": "BIGINT", "F": "FLOAT", "CV": "VARCHAR", "TS": "TIMESTAMP", "I1": "BYTEINT", "I2": "SMALLINT", "D": "DECIMAL"}
cols["type"] = cols.type_code.str.strip().map(TYPES).fillna(cols.type_code)
print("Columns of txn:"); print(cols[["col", "type", "len"]].to_string(index=False))
print("\nFirst 5 rows:"); display(q("SELECT TOP 5 * FROM txn ORDER BY txn_id"))

Columns of txn:
             col    type  len
          txn_id  BIGINT    8
     customer_id  BIGINT    8
          txn_ts VARCHAR 2048
      amount_myr   FLOAT    8
         channel VARCHAR 2048
    merchant_cat VARCHAR 2048
      is_foreign  BIGINT    8
      device_new  BIGINT    8
     hour_of_day  BIGINT    8
    km_from_home   FLOAT    8
    txn_count_1h  BIGINT    8
account_age_days  BIGINT    8
   amt_ratio_30d   FLOAT    8
        is_fraud  BIGINT    8

First 5 rows:


,txn_id,customer_id,txn_ts,amount_myr,channel,merchant_cat,is_foreign,device_new,hour_of_day,km_from_home,txn_count_1h,account_age_days,amt_ratio_30d,is_fraud
0,1,4255,2026-09-12 16:00:00,43.13,DUITNOW,FNB,0,0,1,4.4,0,988,0.42,0
1,2,7994,2026-08-02 10:48:01,36.63,CARD_ECOM,FNB,0,0,12,9.7,0,279,0.91,0
2,3,8447,2026-09-05 22:01:39,63.27,FPX,LUXURY,0,0,6,8.9,2,3607,1.58,0
3,4,17807,2026-07-05 12:47:24,65.75,DUITNOW,FNB,0,0,22,4.5,1,679,1.07,0
4,5,1244,2026-08-27 14:44:14,25.27,CARD_POS,TRAVEL,0,0,22,16.0,1,1420,0.40,0


In [4]:
summary = q("""SELECT COUNT(*) AS transactions, COUNT(DISTINCT customer_id) AS customers,
                      SUM(is_fraud) AS fraud, AVG(CAST(is_fraud AS FLOAT)) AS fraud_rate,
                      MIN(txn_ts) AS first_ts, MAX(txn_ts) AS last_ts,
                      AVG(amount_myr) AS avg_amount, MAX(amount_myr) AS max_amount FROM txn""")
print("All of this was computed inside Teradata (one aggregate query):")
display(summary.T.rename(columns={0: "value"}))

All of this was computed inside Teradata (one aggregate query):


,value
transactions,200000
customers,19997
fraud,2823
fraud_rate,0.014115
first_ts,2026-07-01 00:00:21
last_ts,2026-09-28 23:59:46
avg_amount,110.02704
max_amount,5524.21


In [5]:
by_channel  = q("SELECT channel AS grp, COUNT(*) AS txns, SUM(is_fraud) AS fraud, AVG(CAST(is_fraud AS FLOAT)) AS fraud_rate FROM txn GROUP BY 1 ORDER BY 4 DESC")
by_merchant = q("SELECT merchant_cat AS grp, COUNT(*) AS txns, SUM(is_fraud) AS fraud, AVG(CAST(is_fraud AS FLOAT)) AS fraud_rate FROM txn GROUP BY 1 ORDER BY 4 DESC")
by_hour     = q("SELECT hour_of_day AS grp, COUNT(*) AS txns, SUM(is_fraud) AS fraud, AVG(CAST(is_fraud AS FLOAT)) AS fraud_rate FROM txn GROUP BY 1 ORDER BY 1")
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
for ax, df, title in zip(axes, [by_channel, by_merchant, by_hour], ["Fraud rate by channel", "Fraud rate by merchant category", "Fraud rate by hour of day"]):
    ax.bar(df.grp.astype(str), df.fraud_rate * 100, color="#171e19"); ax.set_title(title); ax.set_ylabel("% fraud")
    ax.tick_params(axis="x", rotation=60)
plt.tight_layout(); plt.show()
print("Reading the charts: night hours (1-5am), CRYPTO/GAMING merchants and e-commerce carry the most fraud; "
      "that is the signal built into the synthetic data and the one the model learns.")
display(by_channel)

<Figure size 1500x360 with 3 Axes>

Reading the charts: night hours (1-5am), CRYPTO/GAMING merchants and e-commerce carry the most fraud; that is the signal built into the synthetic data and the one the model learns.


,grp,txns,fraud,fraud_rate
0,CARD_ECOM,50040,1471,0.029396
1,CARD_POS,69990,654,0.009344
2,ATM,16124,149,0.009241
3,DUITNOW,40070,349,0.008710
4,FPX,23776,200,0.008412


## Step 4. From transactions to model inputs (`txn_features`)
The model needs numbers only. A **view** (no copy of the data) turns categories into 0/1 flags *inside Teradata*, so the
features always match the transactions. This is the exact view definition stored in the database:

In [6]:
view_sql = q("SHOW VIEW txn_features").iloc[0, 0]
print(view_sql.replace("\r", "\n"))
FEATS = ["amount_myr", "is_foreign", "device_new", "hour_of_day", "km_from_home", "txn_count_1h", "amt_ratio_30d",
         "account_age_days", "ch_ecom", "ch_duitnow", "ch_fpx", "ch_atm", "mc_highrisk", "mc_resale", "night_txn"]
print(f"\nThe model uses {len(FEATS)} features:", ", ".join(FEATS))
display(q("SELECT TOP 5 * FROM txn_features ORDER BY txn_id"))

REPLACE VIEW txn_features AS
SELECT txn_id, is_fraud, amount_myr, is_foreign, device_new, hour_of_day, km_from_home,
       txn_count_1h, amt_ratio_30d, account_age_days,
       CASE WHEN channel='CARD_ECOM' THEN 1 ELSE 0 END AS ch_ecom,
       CASE WHEN channel='DUITNOW'   THEN 1 ELSE 0 END AS ch_duitnow,
       CASE WHEN channel='FPX'       THEN 1 ELSE 0 END AS ch_fpx,
       CASE WHEN channel='ATM'       THEN 1 ELSE 0 END AS ch_atm,
       CASE WHEN merchant_cat IN ('CRYPTO','GAMING') THEN 1 ELSE 0 END AS mc_highrisk,
       CASE WHEN merchant_cat IN ('ELECTRONICS','LUXURY') THEN 1 ELSE 0 END AS mc_resale,
       CASE WHEN hour_of_day BETWEEN 1 AND 5 THEN 1 ELSE 0 END AS night_txn
FROM txn

The model uses 15 features: amount_myr, is_foreign, device_new, hour_of_day, km_from_home, txn_count_1h, amt_ratio_30d, account_age_days, ch_ecom, ch_duitnow, ch_fpx, ch_atm, mc_highrisk, mc_resale, night_txn


,txn_id,is_fraud,amount_myr,is_foreign,device_new,hour_of_day,km_from_home,txn_count_1h,amt_ratio_30d,account_age_days,ch_ecom,ch_duitnow,ch_fpx,ch_atm,mc_highrisk,mc_resale,night_txn
0,1,0,43.13,0,0,1,4.4,0,0.42,988,0,1,0,0,0,0,1
1,2,0,36.63,0,0,12,9.7,0,0.91,279,1,0,0,0,0,0,0
2,3,0,63.27,0,0,6,8.9,2,1.58,3607,0,0,1,0,0,1,0
3,4,0,65.75,0,0,22,4.5,1,1.07,679,0,1,0,0,0,0,0
4,5,0,25.27,0,0,22,16.0,1,0.40,1420,0,0,0,0,0,0,0


## Step 5. Where the model and its scores live

* **`fraud_xgb_model`** is the trained model, stored as a table: gradient-boosted decision trees, one row per tree (the whole tree is a JSON document in the last column).
  It never leaves Teradata. Scoring runs there too (`XGBoostPredict`).
* **`model_metrics`** records how that model was trained.
* **`txn_scores`** holds the model's fraud probability for every transaction. The dashboard, the alerts table and the chat all read this table.

In [7]:
model_cols = q("SELECT ColumnName AS col FROM DBC.ColumnsV WHERE DatabaseName = DATABASE AND TableName = 'fraud_xgb_model' ORDER BY ColumnId")
print("fraud_xgb_model columns:", ", ".join(model_cols.col.str.strip()))
shape = q("SELECT COUNT(*) AS trees, COUNT(DISTINCT iter) AS boosting_rounds, COUNT(DISTINCT tree_num) AS ensembles, COUNT(DISTINCT task_index) AS amps_used FROM fraud_xgb_model").iloc[0]
print(f"fraud_xgb_model: {int(shape.trees)} trees are stored. Teradata trains in parallel: {int(shape.amps_used)} AMPs (task_index) each "
      f"trained their own trees on their share of the rows; {int(shape.boosting_rounds)} distinct boosting-round ids (iter) and "
      f"{int(shape.ensembles)} tree_num values appear. Each row's last column is one whole tree as JSON.")
display(q("SELECT TOP 6 * FROM fraud_xgb_model"))
print("\nmodel_metrics (written when the model was trained):")
display(q("SELECT auc, gini, train_rows, test_rows, train_seconds, algorithm FROM model_metrics"))

fraud_xgb_model columns: task_index, tree_num, iter, class_num, tree_order, classification_tree
fraud_xgb_model: 401 trees are stored. Teradata trains in parallel: 4 AMPs (task_index) each trained their own trees on their share of the rows; 51 distinct boosting-round ids (iter) and 3 tree_num values appear. Each row's last column is one whole tree as JSON.


,task_index,tree_num,iter,class_num,tree_order,classification_tree
0,3,1,3,0,0,"{""id_"":1,""sum_"":-0.051259,""sumSq_"":6.499826,""size_"":714,""maxDepth_"":6,""nodeType_"":""REGRESSION_NODE"",""split..."
1,3,1,5,0,0,"{""id_"":1,""sum_"":0.792035,""sumSq_"":5.536158,""size_"":714,""maxDepth_"":6,""nodeType_"":""REGRESSION_NODE"",""split_..."
2,3,1,6,0,0,"{""id_"":1,""sum_"":0.917319,""sumSq_"":4.978474,""size_"":714,""maxDepth_"":6,""nodeType_"":""REGRESSION_NODE"",""split_..."
3,3,1,4,0,0,"{""id_"":1,""sum_"":0.415903,""sumSq_"":6.064627,""size_"":714,""maxDepth_"":6,""nodeType_"":""REGRESSION_NODE"",""split_..."
4,3,1,2,0,0,"{""id_"":1,""sum_"":-1.237580,""sumSq_"":6.751239,""size_"":714,""maxDepth_"":6,""nodeType_"":""REGRESSION_NODE"",""split..."
5,3,1,1,0,0,"{""id_"":1,""sum_"":-3.067400,""sumSq_"":6.944550,""size_"":714,""maxDepth_"":6,""nodeType_"":""REGRESSION_NODE"",""split..."



model_metrics (written when the model was trained):


,auc,gini,train_rows,test_rows,train_seconds,algorithm
0,0.924058,0.848116,160000,40000,7.5,Teradata in-database XGBoost (TD_XGBoost)


In [8]:
bands = q("""SELECT CASE WHEN Prob_1 < 0.1 THEN '0.0-0.1' WHEN Prob_1 < 0.2 THEN '0.1-0.2' WHEN Prob_1 < 0.4 THEN '0.2-0.4'
                         WHEN Prob_1 < 0.6 THEN '0.4-0.6' WHEN Prob_1 < 0.8 THEN '0.6-0.8' WHEN Prob_1 < 0.9 THEN '0.8-0.9 (P2 alert)'
                         ELSE '0.9-1.0 (P1 alert)' END AS band, COUNT(*) AS transactions
             FROM txn_scores GROUP BY 1 ORDER BY 1""")
print("How the 200,000 scores are distributed (computed in Teradata):")
display(bands)
alerts_now = int(q("SELECT COUNT(*) AS n FROM txn_scores WHERE Prob_1 >= 0.8")["n"][0])
print(f"\nAn ALERT is a transaction with probability >= 0.8. Right now that is {alerts_now} of 200,000 "
      f"({alerts_now / 200000:.3%}) transactions.")

How the 200,000 scores are distributed (computed in Teradata):


,band,transactions
0,0.1-0.2,127217
1,0.2-0.4,52829
2,0.4-0.6,16057
3,0.6-0.8,3642
4,0.8-0.9 (P2 alert),242
5,0.9-1.0 (P1 alert),13



An ALERT is a transaction with probability >= 0.8. Right now that is 255 of 200,000 (0.128%) transactions.


## Step 6. What each part of the UI calls in Teradata
Instead of describing it, the cell below **runs the app's own code** for each UI element and **captures the SQL it sends
to Teradata** (by wrapping `db.query_df`). What you see is exactly what happens when you click in the UI.

| UI element | API endpoint | Code that runs |
|---|---|---|
| Dashboard KPI cards + channel chart | `GET /api/kpis` | `service/routers/kpis.py` |
| Model card (AUC, rows, seconds) | `GET /api/model` | `service/routers/ml.py` |
| Alerts table (every row, sorted by probability) | `GET /api/alerts` | `service/routers/alerts.py` |
| Alert detail page (transaction + customer boxes) | `GET /api/alerts/{id}` | `service/data.py` |
| Re-score button | `POST /api/score` | `XGBoostPredict` in-database, result saved to `txn_scores` |
| Probability click: copilot brief | `POST /api/alerts/{id}/brief` | `TD_VectorDistance` search over `policy_emb`, then one LLM call |
| Chat bubble | `POST /api/chat` | read-only SQL tools in `service/services/chat.py` + policy search |
| Documents page | `GET/POST/DELETE /api/documents` | `policy_docs`, `policy_chunks`, `policy_emb` |

In [9]:
from contextlib import contextmanager
from service.routers import alerts as alerts_router, kpis as kpis_router, ml as ml_router
from service.services import chat as chat_service, documents as documents_service, rag

@contextmanager
def capture_sql():
    """Record every SELECT the app code sends to Teradata while the block runs."""
    seen, real = [], db.query_df
    def spy(sql):
        seen.append(" ".join(sql.split()))
        return real(sql)
    db.query_df = spy
    try:
        yield seen
    finally:
        db.query_df = real

def show(title, endpoint, calls, result):
    print("=" * 110); print(f"UI: {title}   ->   {endpoint}"); print("=" * 110)
    for i, sql in enumerate(calls, 1):
        print(f"[Teradata SQL {i}] {sql[:420]}{'...' if len(sql) > 420 else ''}")
    print("[result]", result)
    print()

db.clear_cache()   # the app caches reads for speed; clear it so the real SQL is visible below

with capture_sql() as calls:
    k = kpis_router.kpis()
show("Dashboard KPI cards", "GET /api/kpis", calls,
     {x: k[x] for x in ["total_txn", "fraud_txn", "alerts_open"]} | {"fraud_by_channel": k["fraud_by_channel"][:2], "...": "..."})

with capture_sql() as calls:
    m = ml_router.model_card()
show("Model card", "GET /api/model", calls, {x: m[x] for x in ["auc", "train_rows", "test_rows", "train_seconds"]})

UI: Dashboard KPI cards   ->   GET /api/kpis
[Teradata SQL 1] SELECT channel, COUNT(*) AS txn, SUM(is_fraud) AS fraud FROM txn GROUP BY channel ORDER BY 3 DESC
[Teradata SQL 2] SELECT COUNT(*) AS n FROM txn_scores WHERE Prob_1 >= 0.8
[result] {'total_txn': 200000, 'fraud_txn': 2823, 'alerts_open': 255, 'fraud_by_channel': [{'channel': 'CARD_ECOM', 'txn': 50040, 'fraud': 1471}, {'channel': 'CARD_POS', 'txn': 69990, 'fraud': 654}], '...': '...'}

UI: Model card   ->   GET /api/model
[Teradata SQL 1] SELECT auc, gini, train_rows, test_rows, train_seconds, algorithm, features FROM model_metrics
[result] {'auc': 0.924058000489862, 'train_rows': 160000, 'test_rows': 40000, 'train_seconds': 7.5}



In [10]:
with capture_sql() as calls:
    al = alerts_router.alerts(min_prob=0.0, limit=5000)
show("Alerts table (all rows, highest probability first)", "GET /api/alerts", calls,
     f"{len(al)} rows; first = {al[0]}; last probability = {al[-1]['prob']:.4f}")
top_id = al[0]["txn_id"]

with capture_sql() as calls:
    detail = alerts_router.alert_detail(top_id)
show(f"Alert detail page for transaction #{top_id}", f"GET /api/alerts/{top_id}", calls,
     {"txn": {x: detail["txn"][x] for x in ["amount_myr", "channel", "merchant_cat", "txn_ts"]},
      "prob": round(detail["prob"], 4), "reasons": detail["reasons"], "customer": {x: detail["customer"][x] for x in ["txn_count", "flagged_count", "avg_amount_myr"]}})

UI: Alerts table (all rows, highest probability first)   ->   GET /api/alerts
[Teradata SQL 1] SELECT TOP 5000 t.txn_id, t.amount_myr, t.channel, t.merchant_cat, s.Prob_1 AS prob, t.txn_ts FROM txn t JOIN txn_scores s ON t.txn_id = s.txn_id WHERE s.Prob_1 >= 0.0 ORDER BY s.Prob_1 DESC
[result] 5000 rows; first = {'txn_id': 38067, 'amount_myr': 1028.23, 'channel': 'CARD_ECOM', 'merchant_cat': 'TRAVEL', 'prob': 0.9559516164447363, 'txn_ts': '2026-07-15 06:39:13'}; last probability = 0.5754

UI: Alert detail page for transaction #38067   ->   GET /api/alerts/38067
[Teradata SQL 1] SELECT t.txn_id, t.customer_id, t.txn_ts, t.amount_myr, t.channel, t.merchant_cat, t.is_foreign, t.device_new, t.hour_of_day, t.km_from_home, t.txn_count_1h, t.amt_ratio_30d, t.account_age_days, t.is_fraud, s.Prob_1 FROM txn t JOIN txn_scores s ON t.txn_id = s.txn_id WHERE t.txn_id = 38067
[Teradata SQL 2] SELECT COUNT(*) AS n, AVG(t.amount_myr) AS avg_amt, SUM(t.amount_myr) AS total_amt, MIN(t.txn_ts) AS first_

In [11]:
print("=" * 110); print("UI: Re-score button   ->   POST /api/score   (not run here: it rewrites txn_scores, ~20 s)"); print("=" * 110)
import inspect
print(inspect.getsource(ml_router._rescore)); print()

print("=" * 110); print("UI: Probability click -> copilot brief   ->   POST /api/alerts/{id}/brief   (search step)"); print("=" * 110)
print("[Teradata SQL] the in-database vector search the brief runs for each question (TD_VectorDistance):")
print(rag.VECTOR_SQL.format(lo="<first query id>", hi="<last query id>", dim=rag.DIM, n=rag.CANDIDATES)); print()
question = "How is a suspected fraud case investigated and referred?"
hits = rag.retrieve_many([question], k=3)[0]
print(f"[result] top passages for: {question!r}")
for h in hits: print(f"   {h['doc']} {h['section']:>9}  vector distance {h['distance']:.3f}  blended score {h['score']:.3f}  | {h['text'][:80]!r}")
print("(After this search the brief makes ONE call to the LLM with the retrieved passages; that call is not in Teradata.)\n")

UI: Re-score button   ->   POST /api/score   (not run here: it rewrites txn_scores, ~20 s)
def _rescore() -> dict:
    t0 = time.perf_counter()
    pred = XGBoostPredict(newdata=DataFrame("txn_features"), object=DataFrame("fraud_xgb_model"),
                          id_column="txn_id", model_type="Classification", output_prob=True,
                          output_responses=["0", "1"])
    pred.result.to_sql("txn_scores", if_exists="replace", primary_index="txn_id")
    seconds = round(time.perf_counter() - t0, 2)
    db.clear_cache()  # scores changed: drop cached KPIs and alert lists
    rows = db.query("SELECT COUNT(*) FROM txn_scores")[0][0]
    sql = "-- XGBoostPredict over txn_features, persisted to txn_scores\n" + pred.result.show_query()
    return {"rows": int(rows), "seconds": seconds, "sql": sql}


UI: Probability click -> copilot brief   ->   POST /api/alerts/{id}/brief   (search step)
[Teradata SQL] the in-database vector search the brief runs for each question (TD_Vector

In [12]:
with capture_sql() as calls:
    stats = chat_service.run_tool("alert_stats", {}, [])
    brk = chat_service.run_tool("alerts_breakdown", {"group_by": "channel"}, [])
show("Chat bubble: 'How many transactions need an alert now?'", "POST /api/chat  ->  tool alert_stats", calls[:1], stats)
show("Chat bubble: 'Break down the alerts by channel'", "POST /api/chat  ->  tool alerts_breakdown", calls[1:], brk["groups"][:3])
print("The chat model can ONLY call these fixed tools; it never writes SQL itself:")
print([t["function"]["name"] for t in chat_service.TOOLS])

with capture_sql() as calls:
    docs = documents_service.list_documents()
show("Documents page", "GET /api/documents", calls, docs)

UI: Chat bubble: 'How many transactions need an alert now?'   ->   POST /api/chat  ->  tool alert_stats
[Teradata SQL 1] SELECT COUNT(*) AS n, SUM(CASE WHEN s.Prob_1 >= 0.9 THEN 1 ELSE 0 END) AS p1, SUM(CASE WHEN s.Prob_1 >= 0.8 AND s.Prob_1 < 0.9 THEN 1 ELSE 0 END) AS p2, SUM(t.amount_myr) AS total, AVG(t.amount_myr) AS avg_amt, AVG(s.Prob_1) AS avg_prob FROM txn t JOIN txn_scores s ON t.txn_id = s.txn_id WHERE s.Prob_1 >= 0.8 AND s.Prob_1 <= 1.0
[result] {'alerts': 255, 'priority_1': 13, 'priority_2': 242, 'total_amount_myr': 89897.9, 'average_amount_myr': 352.54, 'average_probability': 0.8427}

UI: Chat bubble: 'Break down the alerts by channel'   ->   POST /api/chat  ->  tool alerts_breakdown
[Teradata SQL 1] SELECT t.channel AS grp, COUNT(*) AS n, SUM(t.amount_myr) AS total, AVG(s.Prob_1) AS avg_prob FROM txn t JOIN txn_scores s ON t.txn_id = s.txn_id WHERE s.Prob_1 >= 0.8 AND s.Prob_1 <= 1.0 GROUP BY 1 ORDER BY 2 DESC
[result] [{'channel': 'CARD_ECOM', 'alerts': 114, 'total_amoun

## Step 7. Consistency checks
The numbers the UI shows must agree with each other and with the raw tables. Any failure stops the notebook.

In [13]:
n_txn   = int(q("SELECT COUNT(*) AS n FROM txn")["n"][0])
n_score = int(q("SELECT COUNT(*) AS n FROM txn_scores")["n"][0])
n_unscored = int(q("SELECT COUNT(*) AS n FROM txn t LEFT JOIN txn_scores s ON t.txn_id = s.txn_id WHERE s.txn_id IS NULL")["n"][0])
fraud = int(q("SELECT SUM(is_fraud) AS n FROM txn")["n"][0])
meta = q("SELECT auc, train_rows, test_rows FROM model_metrics").iloc[0]
chunks = int(q("SELECT COUNT(*) AS n FROM policy_chunks")["n"][0])
embs = int(q("SELECT COUNT(*) AS n FROM policy_emb")["n"][0])
doc_chunks = int(q("SELECT SUM(chunks) AS n FROM policy_docs")["n"][0])

checks = {
    "txn has 200,000 rows": n_txn == 200_000,
    "every transaction has a score (txn_scores = txn, none missing)": n_score == n_txn and n_unscored == 0,
    "KPI card 'total' = txn row count": k["total_txn"] == n_txn,
    "KPI card 'fraud' = SUM(is_fraud)": k["fraud_txn"] == fraud,
    "KPI card 'open alerts' = scores >= 0.8": k["alerts_open"] == alerts_now,
    "alerts table loaded 5,000 rows, sorted by probability (desc)": len(al) == 5000 and all(a["prob"] >= b["prob"] for a, b in zip(al, al[1:])),
    "model card rows add up to the table": int(meta.train_rows + meta.test_rows) == n_txn,
    "model AUC is in the expected range (0.85-1.0)": 0.85 <= float(meta.auc) <= 1.0,
    "every policy passage has exactly one embedding": chunks == embs == doc_chunks > 0,
}
for name, ok in checks.items(): print(("PASS  " if ok else "FAIL  ") + name)
assert all(checks.values()), [n for n, ok in checks.items() if not ok]
print("\nALL CHECKS PASSED")

PASS  txn has 200,000 rows
PASS  every transaction has a score (txn_scores = txn, none missing)
PASS  KPI card 'total' = txn row count
PASS  KPI card 'fraud' = SUM(is_fraud)
PASS  KPI card 'open alerts' = scores >= 0.8
PASS  alerts table loaded 5,000 rows, sorted by probability (desc)
PASS  model card rows add up to the table
PASS  model AUC is in the expected range (0.85-1.0)
PASS  every policy passage has exactly one embedding

ALL CHECKS PASSED
